# CytoTrack AI - public tracking-linker training on Colab\n\nRuntime: **GPU**. This notebook downloads open/public Cell Tracking Challenge training movies, trains the frame-to-frame tracking association model, evaluates baseline vs trained accuracy, and packages the model/report artifacts.\n\nThe local laptop does not need to train the model; Colab handles the heavy run.

In [ ]:
!nvidia-smi || true

In [ ]:
!git clone https://github.com/SciSpectator/CytoTrack-AI.git\n%cd CytoTrack-AI\n!pip install -q -r requirements.txt joblib scikit-learn

In [ ]:
!mkdir -p real_cell_movies\n!wget -q -nc https://data.celltrackingchallenge.net/training-datasets/DIC-C2DH-HeLa.zip -P real_cell_movies\n!wget -q -nc https://data.celltrackingchallenge.net/training-datasets/Fluo-C2DL-Huh7.zip -P real_cell_movies\n!unzip -q -n real_cell_movies/DIC-C2DH-HeLa.zip -d real_cell_movies\n!unzip -q -n real_cell_movies/Fluo-C2DL-Huh7.zip -d real_cell_movies\n!find real_cell_movies -maxdepth 2 -type d | sort | head -30

Train on sequence `01`, hold out sequence `02`. If the trained model is worse than the current baseline, CytoTrack AI keeps the existing tracker and records the regression instead of deploying a worse model.

In [ ]:
!python tools/train_tracking_linker.py \\\n  --trees 300 \\\n  --output-dir RESULT/colab_tracking_linker_training \\\n  --model-dir model_cache/tracking_linker_colab

In [ ]:
import json, pathlib\nreport = json.load(open('RESULT/colab_tracking_linker_training/tracking_linker_report.json'))\nprint(json.dumps(report, indent=2))

In [ ]:
!python tools/self_upgrade_validation_loop.py --iterations 1 --frames-long 180 --stress-window-short 8 --stress-window-long 20

In [ ]:
!zip -r RESULT/tracking_linker_colab_artifacts.zip \\\n  RESULT/colab_tracking_linker_training \\\n  model_cache/tracking_linker_colab \\\n  RESULT/self_upgrade_validation >/dev/null\nprint('Download: RESULT/tracking_linker_colab_artifacts.zip')